# 05 — Train Model v3 (fine-tune từ v2)

## Residual errors sau v2 (camera LIVE)

| Hiện tượng | Hướng v3 |
|------------|----------|
| 5000 → hay thành 20000 | Oversample ảnh chứa 5000/20000 + tăng `cls` + `imgsz` lớn hơn |
| 100000 → hay thành 500000 | Tương tự (cặp polymer gần màu) |
| Để **dọc** khó hơn để **ngang** | Thêm ảnh train xoay 90°/270° + `degrees=90` |
| 1000/2000 & nền rối đã khá | Giữ bằng cách **fine-tune từ v2**, không train lại từ COCO |

Script chính: `../train_v3.py`

## Recipe v3 (tóm tắt)

1. **Init weights:** `notebooks/v2_exports/yolov8s_v2_best.pt`  
2. **Data:** `datasets/vn_currency_yolo_v3` = train gốc + rot90 + rot270 + oversample hard classes  
3. **Train:** imgsz=800, degrees=90, cls=1.5, epochs≈80, batch≈16  
4. **Val/test:** vẫn split Roboflow gốc (không xoay val/test — tránh lạc quan giả)

### Chạy trên server

```bash
rsync -avh --exclude '.venv' --exclude '.git' --exclude 'runs' \
  ~/Desktop/Note/Hoc_Tap/Nhan_dang_mau/nhan_dang_tien_Viet_Nam/ \
  server_02@10.87.25.8:/home/server_02/Documents/dungtv/nhan_dang_tien_Viet_Nam/

source ~/anaconda3/etc/profile.d/conda.sh && conda activate vn_currency
cd ~/Documents/dungtv/nhan_dang_tien_Viet_Nam
mkdir -p logs
# build data lần đầu có thể mất vài phút (xoay toàn bộ train)
nohup python -u train_v3.py --epochs 80 --batch 16 --imgsz 800 > logs/train_v3.log 2>&1 &
tail -f logs/train_v3.log
```

### Copy về local

```bash
rsync -avh server_02@10.87.25.8:/home/server_02/Documents/dungtv/nhan_dang_tien_Viet_Nam/runs/detect/vn_currency_yolov8s_v3/ \
  ~/Desktop/Note/Hoc_Tap/Nhan_dang_mau/nhan_dang_tien_Viet_Nam/runs/detect/vn_currency_yolov8s_v3/

rsync -avh server_02@10.87.25.8:/home/server_02/Documents/dungtv/nhan_dang_tien_Viet_Nam/notebooks/v3_exports/ \
  ~/Desktop/Note/Hoc_Tap/Nhan_dang_mau/nhan_dang_tien_Viet_Nam/notebooks/v3_exports/
```

### Demo

```bash
python demo_camera.py --weights notebooks/v3_exports/yolov8s_v3_best.pt --mode live --conf 0.30 --imgsz 800
```

## Nếu v3 vẫn nhầm 5k↔20k / 100k↔500k trên cam

Ưu tiên **data** hơn hyperparam:
- 30–50 ảnh/class khó: mặt trước+sau, ngang+dọc, nền rối  
- Ảnh **cặp cạnh nhau** 5000 vs 20000, 100000 vs 500000  
- Merge vào Roboflow → export YOLO → train lại (v3.1 fine-tune từ v3)

Tuỳ chọn kỹ thuật sau đó: two-stage verify (crop → classifier) chỉ cho các cặp dễ nhầm.